# Synthetic 11 — Final Aligned Observation Outputs

This notebook finalizes the synthetic stream reconstruction output after the row-rebuild comparison stage.

It supports two output targets:

1. **Full final-aligned observation stage**  
   A lineage-rich table that joins premelt observations with rebuilt sensor observations.

2. **Compact final synthetic output**  
   A smaller Kaggle-like output table intended for downstream inspection, export, or handoff.

The `FINAL_ALIGN_OUTPUT_MODE` setting controls which output is built:

```text
"stage"   = build only the lineage-rich final-aligned stage table
"compact" = build only the compact final output table
"both"    = build both outputs
```

Recommended default for a clean end-to-end run: `both`.

## Ask

Can the rebuilt long-form Kafka/Postgres sensor messages be converted back into a complete observation-level synthetic dataset?

## Answer

This notebook builds the final aligned synthetic output after the rebuild comparison stage. It validates row counts, observation IDs, timestamps, rebuild completeness, and status distributions before the synthetic pipeline moves into Bronze handoff.

## Imports

In [ ]:
import os
import pandas as pd

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
)



from utils.synthetic.pipeline.final_aligned_observation_writer import (
    build_final_aligned_observations_stage,
)

from utils.synthetic.pipeline.final_aligned_output_writer import (
    build_synthetic_final_aligned_output_stage,
)

from utils.core.env_helpers import (
    env_bool,
    
    env_int,
    env_str,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 120)

## Notebook Configurables

Set `FINAL_ALIGN_OUTPUT_MODE` to choose the output behavior for this notebook.

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

FINAL_ALIGN_OUTPUT_MODE = env_str(
    "SYNTHETIC_STAGE_11_OUTPUT_MODE",
    "both",
).strip().lower()

VALID_OUTPUT_MODES = {"stage", "compact", "both"}

if FINAL_ALIGN_OUTPUT_MODE not in VALID_OUTPUT_MODES:
    raise ValueError(
        "SYNTHETIC_STAGE_11_OUTPUT_MODE must be one of "
        f"{sorted(VALID_OUTPUT_MODES)}. Got: {FINAL_ALIGN_OUTPUT_MODE!r}"
    )

IF_EXISTS_FLAG = env_str(
    "SYNTHETIC_FINAL_ALIGN_IF_EXISTS",
    "replace",
).strip().lower()

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_FINAL_ALIGN_COMPLETE_ONLY",
    True,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

PREMELT_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

REBUILT_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

FINAL_ALIGNED_STAGE_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_ALIGNED_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_final_aligned_stage",
    aliases=("FINAL_ALIGNED_STAGE_TABLE_NAME", "FINAL_ALIGNMENT_SOURCE_TABLE_NAME"),
)

FINAL_OUTPUT_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_OUTPUT_TABLE",
    "synthetic_sensor_observations_final_output",
)

TIMESTAMP_OUTPUT_COLUMN = env_str(
    "SYNTHETIC_FINAL_OUTPUT_TIMESTAMP_COLUMN",
    "timestamp",
)

MACHINE_STATUS_OUTPUT_COLUMN = env_str(
    "SYNTHETIC_FINAL_OUTPUT_STATUS_COLUMN",
    "machine_status",
)

print("Synthetic Stage 11 config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("output mode:", FINAL_ALIGN_OUTPUT_MODE)
print("premelt table:", PREMELT_TABLE_NAME)
print("rebuilt table:", REBUILT_TABLE_NAME)
print("final-aligned stage table:", FINAL_ALIGNED_STAGE_TABLE_NAME)
print("compact final output table:", FINAL_OUTPUT_TABLE_NAME)
print("if exists:", IF_EXISTS_FLAG)
print("complete only:", COMPLETE_ONLY_FLAG)
print("observation window size:", OBSERVATION_WINDOW_SIZE)

## SQL Runtime Context

In [ ]:
engine = get_engine_from_env()

## Source Table Smoke Check

This confirms the source tables exist before Stage 11 attempts to build either output table.

In [ ]:
source_table_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema = :schema_name
      AND table_name IN (:premelt_table, :rebuilt_table)
    ORDER BY table_name
    """,
    params={
        "schema_name": SCHEMA,
        "premelt_table": PREMELT_TABLE_NAME,
        "rebuilt_table": REBUILT_TABLE_NAME,
    },
)

display(source_table_check_dataframe)

found_source_tables = set(source_table_check_dataframe["table_name"].tolist())
required_source_tables = {PREMELT_TABLE_NAME, REBUILT_TABLE_NAME}
missing_source_tables = sorted(required_source_tables - found_source_tables)

if missing_source_tables:
    raise RuntimeError(f"Missing Stage 11 source table(s): {missing_source_tables}")

## Expected Row Count

The expected row count is calculated dynamically from the rebuilt observation table instead of hardcoding `225000`.

In [ ]:
expected_counts_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS rebuilt_complete_count,
        COUNT(DISTINCT observation_index) AS rebuilt_distinct_observation_count,
        COUNT(*) FILTER (WHERE observation_index IS NULL) AS rebuilt_null_observation_index_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index
    FROM "{SCHEMA}"."{REBUILT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(expected_counts_dataframe)

expected_counts = expected_counts_dataframe.iloc[0].to_dict()

EXPECTED_ROW_COUNT = int(
    expected_counts["rebuilt_complete_count"]
    if COMPLETE_ONLY_FLAG
    else expected_counts["rebuilt_row_count"]
)

if EXPECTED_ROW_COUNT <= 0:
    raise RuntimeError(
        "Expected row count is zero. Confirm Stage 09/10 rebuilt observations "
        "exist for the active dataset_id and run_id."
    )

print("Expected Stage 11 row count:", EXPECTED_ROW_COUNT)

## Build Full Final-Aligned Observation Stage

This output preserves lineage and reconstruction metadata. It is the best source for debugging, validation, and Bronze handoff checks.

In [ ]:
stage_11a_result = None

if FINAL_ALIGN_OUTPUT_MODE in {"stage", "both"}:
    stage_11a_result = build_final_aligned_observations_stage(
        engine=engine,
        schema=SCHEMA,
        premelt_table=PREMELT_TABLE_NAME,
        rebuilt_table=REBUILT_TABLE_NAME,
        target_table=FINAL_ALIGNED_STAGE_TABLE_NAME,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        n_sensors=NUMBER_OF_SENSORS,
        complete_only=COMPLETE_ONLY_FLAG,
        prefer_rebuilt_sensor_values=True,
        if_exists=IF_EXISTS_FLAG,
        observation_window_size=OBSERVATION_WINDOW_SIZE,
    )

    display(pd.DataFrame([stage_11a_result]))
else:
    print("Skipped full final-aligned observation stage build.")

## Validate Full Final-Aligned Observation Stage

In [ ]:
stage_11a_validation_dataframe = None

if FINAL_ALIGN_OUTPUT_MODE in {"stage", "both"}:
    stage_11a_validation_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS final_row_count,
            COUNT(DISTINCT observation_index) AS distinct_observation_count,
            COUNT(DISTINCT generated_row_id) AS distinct_generated_row_id_count,
            COUNT(*) FILTER (WHERE observation_index IS NULL) AS null_observation_index_count,
            COUNT(*) FILTER (WHERE generated_row_id IS NULL) AS null_generated_row_id_count,
            COUNT(*) FILTER (WHERE observation_timestamp IS NULL) AS null_observation_timestamp_count,
            COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS rebuild_complete_count,
            MIN(observation_index) AS min_observation_index,
            MAX(observation_index) AS max_observation_index,
            MIN(observation_timestamp) AS min_observation_timestamp,
            MAX(observation_timestamp) AS max_observation_timestamp
        FROM "{SCHEMA}"."{FINAL_ALIGNED_STAGE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    stage_11a_validation_dataframe["ready_for_bronze_handoff_check"] = (
        (stage_11a_validation_dataframe["final_row_count"] == EXPECTED_ROW_COUNT)
        & (stage_11a_validation_dataframe["distinct_observation_count"] == EXPECTED_ROW_COUNT)
        & (stage_11a_validation_dataframe["distinct_generated_row_id_count"] == EXPECTED_ROW_COUNT)
        & (stage_11a_validation_dataframe["null_observation_index_count"] == 0)
        & (stage_11a_validation_dataframe["null_generated_row_id_count"] == 0)
        & (stage_11a_validation_dataframe["null_observation_timestamp_count"] == 0)
        & (stage_11a_validation_dataframe["rebuild_complete_count"] == EXPECTED_ROW_COUNT)
    )

    display(stage_11a_validation_dataframe)
else:
    print("Skipped full final-aligned observation stage validation.")

## Build Compact Final Synthetic Output

This output is a compact, Kaggle-like observation table with one row per observation and one column per sensor.

In [ ]:
stage_11b_result = None

if FINAL_ALIGN_OUTPUT_MODE in {"compact", "both"}:
    stage_11b_result = build_synthetic_final_aligned_output_stage(
        engine=engine,
        schema=SCHEMA,
        rebuilt_table=REBUILT_TABLE_NAME,
        target_table=FINAL_OUTPUT_TABLE_NAME,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        n_sensors=NUMBER_OF_SENSORS,
        complete_only=COMPLETE_ONLY_FLAG,
        if_exists=IF_EXISTS_FLAG,
        observation_window_size=OBSERVATION_WINDOW_SIZE,
        timestamp_output_column=TIMESTAMP_OUTPUT_COLUMN,
        machine_status_output_column=MACHINE_STATUS_OUTPUT_COLUMN,
    )

    display(pd.DataFrame([stage_11b_result]))
else:
    print("Skipped compact final synthetic output build.")

## Validate Compact Final Synthetic Output

In [ ]:
stage_11b_validation_dataframe = None

if FINAL_ALIGN_OUTPUT_MODE in {"compact", "both"}:
    stage_11b_validation_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT dataset_id) AS dataset_id_count,
            COUNT(DISTINCT run_id) AS run_id_count,
            COUNT(DISTINCT asset_id) AS asset_id_count,
            COUNT(*) FILTER (WHERE "{TIMESTAMP_OUTPUT_COLUMN}" IS NULL) AS null_timestamp_count,
            COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" IS NULL) AS null_machine_status_count,
            MIN("{TIMESTAMP_OUTPUT_COLUMN}") AS min_timestamp,
            MAX("{TIMESTAMP_OUTPUT_COLUMN}") AS max_timestamp,
            COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'NORMAL') AS normal_rows,
            COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'BROKEN') AS broken_rows,
            COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'RECOVERING') AS recovering_rows
        FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    stage_11b_validation_dataframe["ready_for_export"] = (
        (stage_11b_validation_dataframe["row_count"] == EXPECTED_ROW_COUNT)
        & (stage_11b_validation_dataframe["null_timestamp_count"] == 0)
        & (stage_11b_validation_dataframe["null_machine_status_count"] == 0)
    )

    display(stage_11b_validation_dataframe)
else:
    print("Skipped compact final synthetic output validation.")

## Status Distribution

In [ ]:
if FINAL_ALIGN_OUTPUT_MODE in {"stage", "both"}:
    stage_11a_status_distribution_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            stream_state,
            phase,
            COUNT(*) AS row_count
        FROM "{SCHEMA}"."{FINAL_ALIGNED_STAGE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        GROUP BY stream_state, phase
        ORDER BY stream_state, phase
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Full final-aligned stage status distribution")
    display(stage_11a_status_distribution_dataframe)

if FINAL_ALIGN_OUTPUT_MODE in {"compact", "both"}:
    stage_11b_status_distribution_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            "{MACHINE_STATUS_OUTPUT_COLUMN}" AS machine_status,
            COUNT(*) AS row_count
        FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        GROUP BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
        ORDER BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Compact final output status distribution")
    display(stage_11b_status_distribution_dataframe)

## Sample Output

In [ ]:
if FINAL_ALIGN_OUTPUT_MODE in {"stage", "both"}:
    stage_11a_sample_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            dataset_id,
            run_id,
            asset_id,
            generated_row_id,
            observation_index,
            observation_timestamp,
            stream_state,
            phase,
            sensor_00,
            sensor_01,
            sensor_02,
            sensor_03,
            sensor_04,
            rebuild_sensor_count,
            rebuild_is_complete
        FROM "{SCHEMA}"."{FINAL_ALIGNED_STAGE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        ORDER BY observation_index
        LIMIT 10
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Full final-aligned stage sample")
    display(stage_11a_sample_dataframe)

if FINAL_ALIGN_OUTPUT_MODE in {"compact", "both"}:
    stage_11b_sample_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            dataset_id,
            run_id,
            asset_id,
            "{TIMESTAMP_OUTPUT_COLUMN}",
            sensor_00,
            sensor_01,
            sensor_02,
            sensor_03,
            sensor_04,
            "{MACHINE_STATUS_OUTPUT_COLUMN}"
        FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        ORDER BY "{TIMESTAMP_OUTPUT_COLUMN}", dataset_id, run_id, asset_id
        LIMIT 10
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Compact final output sample")
    display(stage_11b_sample_dataframe)

## Final Stage 11 Summary

In [ ]:
summary_rows = []

if stage_11a_result is not None:
    stage_ready = bool(stage_11a_validation_dataframe["ready_for_bronze_handoff_check"].iloc[0])
    summary_rows.append(
        {
            "output": "full_final_aligned_stage",
            "table_name": FINAL_ALIGNED_STAGE_TABLE_NAME,
            "expected_rows": EXPECTED_ROW_COUNT,
            "actual_rows": int(stage_11a_validation_dataframe["final_row_count"].iloc[0]),
            "ready": stage_ready,
        }
    )

if stage_11b_result is not None:
    compact_ready = bool(stage_11b_validation_dataframe["ready_for_export"].iloc[0])
    summary_rows.append(
        {
            "output": "compact_final_output",
            "table_name": FINAL_OUTPUT_TABLE_NAME,
            "expected_rows": EXPECTED_ROW_COUNT,
            "actual_rows": int(stage_11b_validation_dataframe["row_count"].iloc[0]),
            "ready": compact_ready,
        }
    )

stage_11_summary_dataframe = pd.DataFrame(summary_rows)
display(stage_11_summary_dataframe)

if not stage_11_summary_dataframe.empty and not stage_11_summary_dataframe["ready"].all():
    raise RuntimeError("Stage 11 completed, but one or more outputs failed validation.")

print("Stage 11 complete.")

## Cleanup

In [ ]:
engine.dispose()